# Lab 5 (CAPSTONE): Build the Shopping Assistant Agent with Strands + Amazon Bedrock AgentCore

In this capstone you build the **same** shopping assistant as before — integrated with a
Knowledge Base, an action-group Lambda (price lookup + calculator), and a Guardrail — but
instead of clicking **Create agent** in the Bedrock console, you define the agent in code with
the **Strands Agents SDK** and deploy it to **Amazon Bedrock AgentCore Runtime** as a managed
endpoint.

**Architecture**
```
Shopper question
    └► Shopping Assistant Agent   (AgentCore Runtime — Nova 2 Lite + Guardrail)
              ├► search_products   → shopping-kb (Bedrock Knowledge Base)
              ├► price_lookup      → ShoppingAssistantFunction (Lambda)
              └► calculator        → ShoppingAssistantFunction (Lambda)
```

You edit the agent code in `agents/shopping_assistant.py` (Tasks 2–5), then run the cells
below to deploy and test it. A completed reference is in
`completed_agents/shopping_assistant_full.py` if you get stuck.

## Task 6.1 — Setup

In [ ]:
import boto3
import json
import os
import time
import uuid
import zipfile
import subprocess

REGION     = "us-east-1"
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
agentcore_runtime = boto3.client("bedrock-agentcore",         region_name=REGION)
cfn_client        = boto3.client("cloudformation",            region_name=REGION)

print(f"Account: {ACCOUNT_ID}")

Retrieve the lab resource IDs from the CloudFormation stack outputs. The lab stack is
identified by having `AgentCoreExecutionRoleArn` in its outputs.

In [ ]:
paginator = cfn_client.get_paginator("describe_stacks")
outputs = {}
for page in paginator.paginate():
    for stack in page["Stacks"]:
        stack_outputs = {o["OutputKey"]: o["OutputValue"] for o in stack.get("Outputs", [])}
        if "AgentCoreExecutionRoleArn" in stack_outputs:
            outputs = stack_outputs
            print(f"Found lab stack: {stack['StackName']}")
            break
    if outputs:
        break

if not outputs:
    raise RuntimeError("Could not find the lab stack. Make sure the stack deployed successfully.")

EXECUTION_ROLE_ARN = outputs["AgentCoreExecutionRoleArn"]
KNOWLEDGE_BASE_ID  = outputs["KnowledgeBaseId"]
GUARDRAIL_ID       = outputs["GuardrailId"]
GUARDRAIL_VERSION  = outputs["GuardrailVersion"]
S3_BUCKET          = outputs["DeployBucketName"]
TOOL_LAMBDA_NAME   = outputs["ToolLambdaName"]

print(f"Execution Role : {EXECUTION_ROLE_ARN}")
print(f"Knowledge Base : {KNOWLEDGE_BASE_ID}")
print(f"Guardrail      : {GUARDRAIL_ID} (v{GUARDRAIL_VERSION})")
print(f"Deploy Bucket  : {S3_BUCKET}")
print(f"Tool Lambda    : {TOOL_LAMBDA_NAME}")

## Helper: package and deploy an agent to AgentCore Runtime

This helper installs arm64-compatible dependencies (required by AgentCore Runtime), injects
the lab resource IDs as environment variables into the agent, packages everything as a `.zip`,
uploads it to S3, and creates the AgentCore Runtime.

In [ ]:
def deploy_agent(agent_name: str, agent_file: str, env_vars: dict = None) -> str:
    """Package agent_file + deps, upload to S3, create an AgentCore Runtime, and
    wait until it is READY. Returns the agentRuntimeArn."""
    print(f"\n--- Deploying {agent_name} ---")
    zip_path = f"/tmp/{agent_name}.zip"
    pkg_dir  = f"/tmp/{agent_name}_pkg"
    print("Installing dependencies for arm64...")
    subprocess.run([
        "uv", "pip", "install",
        "--python-platform", "aarch64-manylinux2014",
        "--python-version", "3.13",
        f"--target={pkg_dir}",
        "--only-binary=:all:",
        "bedrock-agentcore", "strands-agents", "boto3",
    ], check=True)
    agent_src = open(f"agents/{agent_file}").read()
    if env_vars:
        env_block = "\n".join(
            f"os.environ.setdefault('{k}', '{v}')" for k, v in env_vars.items()
        )
        agent_src = f"import os\n{env_block}\n\n" + agent_src
    # AgentCore requires the entrypoint file to be named entrypoint.py
    with open(f"{pkg_dir}/entrypoint.py", "w") as f:
        f.write(agent_src)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(pkg_dir):
            for file in files:
                fp = os.path.join(root, file)
                zf.write(fp, os.path.relpath(fp, pkg_dir))
    s3_key = f"{agent_name}/deployment_package.zip"
    print(f"Uploading to s3://{S3_BUCKET}/{s3_key}")
    boto3.client("s3", region_name=REGION).upload_file(
        zip_path, S3_BUCKET, s3_key,
        ExtraArgs={"ExpectedBucketOwner": ACCOUNT_ID},
    )
    print("Creating AgentCore Runtime...")
    resp = agentcore_control.create_agent_runtime(
        agentRuntimeName=agent_name,
        agentRuntimeArtifact={
            "codeConfiguration": {
                "code": {"s3": {"bucket": S3_BUCKET, "prefix": s3_key}},
                "runtime": "PYTHON_3_13",
                "entryPoint": ["entrypoint.py"],
            }
        },
        networkConfiguration={"networkMode": "PUBLIC"},
        roleArn=EXECUTION_ROLE_ARN,
        lifecycleConfiguration={"idleRuntimeSessionTimeout": 300, "maxLifetime": 1800},
    )
    arn        = resp["agentRuntimeArn"]
    runtime_id = arn.split("/")[-1]
    print(f"Created: {arn}  (status: {resp['status']})")
    # Wait until the runtime is invocable. create_agent_runtime returns while the
    # runtime is still CREATING, so poll until it is READY before returning.
    print("Waiting for the runtime to become READY (this can take 1-3 minutes)...")
    for _ in range(60):  # up to ~5 minutes
        status = agentcore_control.get_agent_runtime(
            agentRuntimeId=runtime_id
        )["status"]
        if status == "READY":
            print(f"Runtime is READY: {arn}")
            return arn
        if status in ("CREATE_FAILED", "UPDATE_FAILED", "DELETING"):
            raise RuntimeError(
                f"Runtime {agent_name} entered status {status}. "
                f"Delete it in the Cleanup section, fix agents/{agent_file}, and redeploy."
            )
        print(f"  status: {status} ... waiting")
        time.sleep(5)
    raise TimeoutError(
        f"Runtime {agent_name} did not become READY in time. "
        f"Check its status and re-run, or delete it in the Cleanup section."
    )

def invoke_agent(agent_arn: str, prompt: str, session_id: str = None) -> str:
    """Invoke a deployed AgentCore Runtime and return the text result."""
    resp = agentcore_runtime.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        qualifier="DEFAULT",
        runtimeSessionId=session_id or str(uuid.uuid4()),
        payload=json.dumps({"prompt": prompt}),
    )
    raw = resp["response"].read()
    return json.loads(raw).get("result", str(raw))

## Task 6.2 — Verify the Knowledge Base

Confirm the pre-created `shopping-kb` knowledge base is `ACTIVE` before deploying the agent.

In [ ]:
bedrock_agent_client = boto3.client("bedrock-agent", region_name=REGION)
kb_detail = bedrock_agent_client.get_knowledge_base(knowledgeBaseId=KNOWLEDGE_BASE_ID)
print(f"Name  : {kb_detail['knowledgeBase']['name']}")
print(f"Status: {kb_detail['knowledgeBase']['status']}")
# Expected: Status = ACTIVE

## Task 6.3 — Deploy the Shopping Assistant Agent

> **Before running this cell:** complete `agents/shopping_assistant.py` (Tasks 2–5) — add the
> three tool functions, attach the guardrail to the model, set the system prompt, and create
> the `agent`.
>
> ⚠️ **Important:** you must finish the agent file first. Deploying incomplete code creates a
> broken runtime that you must delete (Cleanup section) before redeploying.

The Knowledge Base ID, Guardrail ID/version, and tool Lambda name are injected as environment
variables so the deployed agent picks up this account's resources automatically.
The deploy cell **waits until the runtime is `READY`** before finishing, so you can safely run Task 6.4 as soon as it returns.

In [ ]:
SHOPPING_ASSISTANT_ARN = deploy_agent(
    agent_name="shopping_assistant_agent",
    agent_file="shopping_assistant.py",
    env_vars={
        "KNOWLEDGE_BASE_ID": KNOWLEDGE_BASE_ID,
        "GUARDRAIL_ID":      GUARDRAIL_ID,
        "GUARDRAIL_VERSION": GUARDRAIL_VERSION,
        "TOOL_LAMBDA_NAME":  TOOL_LAMBDA_NAME,
    },
)
print(f"\nShopping Assistant ARN: {SHOPPING_ASSISTANT_ARN}")

## Task 6.4 — Test the agent

Reuse the same `runtimeSessionId` across the calls below so the agent keeps short-term
conversational context within the session — the equivalent of the console agent's session memory.

> **Note:** This lab uses a non-production account with a model invocation rate limit. If a cell
> returns a **Throttling** error, wait up to 60 seconds and rerun it.

In [ ]:
SESSION_ID = f"shopping-session-{uuid.uuid4()}"

# Knowledge Base question
print("=== KB question ===")
print(invoke_agent(SHOPPING_ASSISTANT_ARN,
    "Hello, I am interested in your products. Who manufactures the products you sell?",
    session_id=SESSION_ID))

In [ ]:
# Price lookup (uses the price_lookup tool -> Lambda)
print("=== Price lookup ===")
print(invoke_agent(SHOPPING_ASSISTANT_ARN,
    "How much does a string trimmer cost?",
    session_id=SESSION_ID))

In [ ]:
# Calculation (uses the calculator tool -> Lambda)
print("=== Calculation ===")
print(invoke_agent(SHOPPING_ASSISTANT_ARN,
    "How much will it cost to buy 2 string trimmers?",
    session_id=SESSION_ID))

In [ ]:
# Guardrail (should be blocked - lawn maintenance advice)
print("=== Guardrail test ===")
print(invoke_agent(SHOPPING_ASSISTANT_ARN,
    "Can I use a string trimmer to control weeds?",
    session_id=SESSION_ID))
# Expected: the guardrail blocks the response with the configured message.

## Cleanup

Delete the AgentCore Runtime when you are done to avoid ongoing charges.

In [ ]:
runtime_id = SHOPPING_ASSISTANT_ARN.split("/")[-1]
agentcore_control.delete_agent_runtime(agentRuntimeId=runtime_id)
print("Deleted: shopping_assistant_agent")

### Conclusion

You built the shopping assistant agent in code with the Strands Agents SDK, attached the
Knowledge Base, action-group Lambda tools, and Guardrail, and deployed it to Amazon Bedrock
AgentCore Runtime — then invoked it as a managed endpoint. Close this notebook and return to
the lab guide to finish.